# Data cleaning and temporal split

This notebook reads the authorized IEEE-CIS training CSVs, normalizes their column names, joins identity fields to transactions, adds relative-time and amount features, and creates an 80/20 chronological split.

Missing feature values are preserved for later train-only imputation. The exported `clean_test.parquet` is a held-out slice of the labeled training data, not Kaggle's separate competition test set.

Outputs are written to `data-science/data/processed/clean_train.parquet` and `clean_test.parquet`.

In [2]:
from pathlib import Path
import sys

import pandas as pd

notebook_dir = Path.cwd().resolve()
repository_root = next(
    (path for path in (notebook_dir, *notebook_dir.parents)
     if (path / "data-science" / "src").is_dir()),
    None,
)
if repository_root is None:
    raise RuntimeError("Could not locate the repository root from the notebook directory.")

data_science_dir = repository_root / "data-science"
sys.path.insert(0, str(data_science_dir / "src"))
from preprocessing.cleaning import clean_ieee_cis_data, temporal_train_test_split

raw_dir = data_science_dir / "data" / "raw"
transaction_path = raw_dir / "train_transaction.csv" / "train_transaction.csv"
identity_path = raw_dir / "train_identity.csv" / "train_identity.csv"

print("Loading IEEE-CIS CSV files...")
df_transactions = pd.read_csv(transaction_path, low_memory=False)
df_identity = pd.read_csv(identity_path, low_memory=False)
print("Transactions:", df_transactions.shape)
print("Identity records:", df_identity.shape)

print("Cleaning, joining identity records, and adding features...")
df_clean = clean_ieee_cis_data(df_transactions, df_identity)
clean_train, clean_test = temporal_train_test_split(df_clean, train_fraction=0.80)

assert len(clean_train) + len(clean_test) == len(df_clean)
assert clean_train["transaction_dt"].max() < clean_test["transaction_dt"].min()

processed_dir = data_science_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
train_path = processed_dir / "clean_train.parquet"
test_path = processed_dir / "clean_test.parquet"
clean_train.to_parquet(train_path, index=False)
clean_test.to_parquet(test_path, index=False)

print("Clean train:", clean_train.shape, "fraud rate:", round(clean_train["is_fraud"].mean(), 4))
print("Clean test:", clean_test.shape, "fraud rate:", round(clean_test["is_fraud"].mean(), 4))
print("Train time range:", clean_train["transaction_dt"].min(), "to", clean_train["transaction_dt"].max())
print("Test time range:", clean_test["transaction_dt"].min(), "to", clean_test["transaction_dt"].max())
print("Exported:", train_path)
print("Exported:", test_path)

Loading IEEE-CIS CSV files...
Transactions: (590540, 394)
Identity records: (144233, 41)
Cleaning, joining identity records, and adding features...
Clean train: (472432, 438) fraud rate: 0.0351
Clean test: (118108, 438) fraud rate: 0.0344
Train time range: 86400 to 12192842
Test time range: 12192900 to 15811131
Exported: D:\Masters_Degree_Fraud_Detection\credit-card-fraud-detection\data-science\data\processed\clean_train.parquet
Exported: D:\Masters_Degree_Fraud_Detection\credit-card-fraud-detection\data-science\data\processed\clean_test.parquet
